# Lab02 — Deploy Nova Assistant to Agent Runtime and register it

**Storyline.** The prototype convinced the support team. Now Nova Assistant must run 24/7
for real shoppers, keep conversations across requests, and be discoverable by the rest of
the company.

**You will learn**
1. What **Agent Runtime** is and what `agents-cli deploy` actually does
2. How to call a deployed agent three ways: CLI, Python SDK, and the **A2A** protocol
3. How server-side **Sessions** make conversations persistent without any code
4. What **Agent Registry** is and why your agent is already in it

Estimated time: 30 minutes (one 5–10 minute deployment).

> **Terminal or notebook — your choice.** Every cell that calls a CLI prints the exact command first (`$ …`).
> Copy it into your own terminal (from the repo root) if you prefer to run it yourself; the cells just automate the same commands.

In [ ]:
# --- Workshop configuration (same cell at the top of every lab) ---
import os, sys, json, pathlib
from dotenv import load_dotenv

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID      = os.environ["PROJECT_ID"]
PROJECT_NUMBER  = os.environ["PROJECT_NUMBER"]
REGION          = os.environ["REGION"]           # europe-west1: Agent Runtime, Sessions, Memory Bank, Gateway, Model Armor
MODEL_LOCATION  = os.environ["MODEL_LOCATION"]   # eu: multi-region endpoint that serves gemini-3.8-flash
MODEL           = os.environ["MODEL"]            # gemini-3.8-flash
AGENT_NAME      = os.environ["AGENT_NAME"]       # nova-assistant
AGENT_DIR       = REPO_ROOT / AGENT_NAME

# Make every shell (!) and SDK call in this notebook target the workshop project.
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION}")
print(f"Agent dir: {AGENT_DIR}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")

## 2.1 Agent Runtime in two minutes

[**Agent Runtime**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime) (the service formerly called *Vertex AI Agent Engine*; the API still says
`reasoningEngines`) is the managed, serverless runtime of Gemini Enterprise Agent Platform:

* You give it a **container** (the project's `Dockerfile`) — Agent Runtime builds and runs it, scales it between a configurable minimum and maximum number of instances, and fronts it with an authenticated HTTPS API.
* Every deployment is also a **context store**: **Sessions** (conversation history + state), **Memory Bank** (long-term memories) and **Code Execution sandboxes** live on the same resource. ADK picks them up automatically.
* Telemetry (traces, logs, metrics) is switched on by the CLI by default (Lab04).
* Optionally it gives the agent its own **Agent Identity** and routes its traffic through an **Agent Gateway** (Lab06).

### FAQ from CEE customers

| Question | Answer |
| --- | --- |
| **What runs my code?** | Your **container**, hosted in a Google-owned **tenant project** on Google infrastructure outside your VPC — one tenant per customer project, reachable from your VPC only through Private Service Connect. Anything **untrusted** (LLM-generated code, browsing) belongs in **Sandboxes** with a strong security boundary (Lab03). Data at rest: CMEK optional; network: VPC Service Controls supported. |
| **How is it billed?** | Compute **$0.085 per vCPU-hour** and memory **$0.009 per GiB-hour**, rounded to the **second**. *"For Runtime, idle time spent waiting for the next prompt between turns is not billed."* You are billed while a turn is being processed (model and tool calls included), not while the instance sits waiting for the next request. Sessions and Memory Bank add small storage/operation charges. |
| **How long may one request run?** | A streaming query (`stream_query`, what ADK uses) may run **15 minutes**; a bidirectional stream **10 minutes**. For longer work there are **long-running query jobs**: `asyncQuery` runs the same agent asynchronously for **up to 7 days** and writes the result to Cloud Storage (instances created after April 22, 2026). |
| **How does it scale?** | `min_instances` 0–10 (default 1), `max_instances` 1–1000 (default 100), per-container `cpu` 1–8 / `memory` 1–32 GiB (default 4 vCPU / 4 GiB), `container_concurrency` (default 9). `agents-cli deploy` uses the defaults; change them in the console or with the SDK. |

### Related services - To be covered in later labs

Everything below is part of Gemini Enterprise Agent Platform and works with or lives on an Agent Runtime instance. Two are **Preview**.

| Service | What it is | In this workshop |
| --- | --- | --- |
| [**Sessions**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/sessions) | Managed conversation store: the events and `state` of one conversation, per user, with a TTL. ADK's `VertexAiSessionService` reads and writes it; a deployed agent uses it automatically. | Lab02 (deployed agent), Lab04 (local development too) |
| [**Memory Bank**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/memory-bank) | Long-term memory across sessions: Gemini extracts facts about a user from finished sessions and returns the relevant ones on the next conversation. | Lab04 |
| [**Sandboxes**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/sandbox) | Isolated environments spawned on demand for untrusted work: code execution, computer use, custom containers, snapshots. Documented as *secure container sandboxing* — this is where LLM-generated code belongs, not in the runtime. | Lab04 (Code Execution as a tool) |
| [**Feedback service**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/feedback-service) — **Preview** | Managed store for end-user thumbs-up/down with labels and free text, bound to one session + response event; shown next to the trace in the console. A companion API to the runtime, not a deploy option. | Lab04 |
| [**Autonomous agent scheduling**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/efficiency/autonomous-scheduling) — **Preview** | The *deferred tier* of the Interactions API: latency-tolerant tasks are queued to off-peak capacity at a 50 % model discount, 95 % finished within 24 h. Today it supports only Google's managed **Deep Research Agent**, not agents you deploy on Agent Runtime; the runtime-native way to run long background work is the long-running query job from the FAQ. | Not used (not applicable to custom agents yet) |

`agents-cli deploy` reads `agents-cli-manifest.yaml` (target = `agent_runtime`, region =
`europe-west1`), packages the project, propagates your `.env` (that's how `GOOGLE_CLOUD_LOCATION=eu`
reaches the cloud) and creates — or updates, matched by display name — the Agent Runtime instance.

## 2.2 Deploy

Run the deployment. The first build takes **5–10 minutes**; updates are faster. If the
cell is interrupted, the deployment continues server-side — check it with
`agents-cli deploy --status` from the project directory.

We add `--agent-identity`: the agent gets its **own cryptographic identity** (a SPIFFE
principal such as `principal://agents.global.org-…/reasoningEngines/ID`) instead of running
as a shared service account. It costs nothing now, and it is a prerequisite for Agent Gateway
policies in Lab06 — and it can only be set when the instance is **created**.

### Agent Identity and the EU model endpoint — how we keep processing in the EU

Agent Identity hardens the agent's credentials: its access tokens are **certificate-bound**, and the Google SDKs inside the
container call the **mTLS** API hosts (`*.mtls.googleapis.com`). Those hosts exist for single regions
(`europe-west1-aiplatform.mtls.googleapis.com`) and for `global`; the `eu` multi-region endpoint
(`aiplatform.eu.rep.googleapis.com`) is reached without token binding today. Gemini 3.8 Flash is served from `eu` and
`global` (September 2026; Gemini 3.5 Flash additionally from `europe-west3`, Gemini 2.5 Flash from most EU regions).

That gives three valid EU setups, and you pick by priority:

| Priority | Setup | What you get |
| --- | --- | --- |
| newest model **and** EU processing (this workshop) | Gemini 3.8 Flash on `eu`, Agent Identity on, token binding off | EU data residency for model calls, the agent's own identity for every IAM and gateway policy |
| certificate-bound tokens **and** EU processing | Gemini 3.5 Flash on `europe-west3`, everything on | regional endpoint, mTLS end to end |
| certificate-bound tokens with the newest model | Gemini 3.8 Flash on `global`, everything on | mTLS end to end, processing not pinned to the EU |

Our choice is the first row. It is a two-line `.env` setting straight from Google's Agent Identity guide —
`GOOGLE_API_USE_CLIENT_CERTIFICATE=false` and `GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false` — which the
CLI propagates to the deployment. The agent keeps its identity, every IAM and gateway policy in Lab06 applies unchanged; only the
token-binding hardening waits. It only matters in the cloud: your laptop has no agent certificate, which is why Lab01 ran as is.
An mTLS host for the multi-region endpoints has been requested from the product team; when it ships, remove the two lines,
redeploy, and token binding is on with nothing else to change.

In [ ]:
# --- Prepare nova-assistant/.env for the cloud: two settings the deployed container needs ---
env_path = AGENT_DIR / ".env"
extra = ""

# Setting 1: let the agent reach the eu model endpoint while running under an Agent Identity.
# Agent Identity binds tokens to a client certificate, and the eu endpoint has no mTLS host yet; opting out keeps residency.
if "GOOGLE_API_USE_CLIENT_CERTIFICATE" not in env_path.read_text():
    extra += ("\n# Agent Identity + eu model endpoint (no mTLS host yet): opt out of client certs / certificate-bound tokens\n"
              "GOOGLE_API_USE_CLIENT_CERTIFICATE=false\nGOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false\n")

# Setting 2: stop uv from re-syncing dependencies on every cold start (no internet behind the Lab06 gateway).
if "UV_NO_SYNC" not in env_path.read_text():
    extra += ("\n# The container starts with `uv run uvicorn ...`; without this, uv may try to (re)build the project from PyPI\n"
              "# on every cold start. In Lab06 the agent sits behind a default-deny gateway, so start-up must not need the internet.\n"
              "UV_NO_SYNC=1\n")

# Append whatever is still missing (re-running the cell adds nothing twice) and show the result.
env_path.write_text(env_path.read_text().rstrip() + "\n" + extra)
print(env_path.read_text())


In [ ]:
# --- Deploy the agent to Agent Runtime with its own Agent Identity (takes a few minutes) ---
import subprocess, json, time

# Run `agents-cli deploy`: packages nova-assistant, builds the container, creates the Agent Runtime instance.
# --agent-identity gives the agent its own principal (immutable after creation); --no-confirm-project skips the prompt.
cmd = f"agents-cli deploy --project {PROJECT_ID} --region {REGION} --agent-identity --no-confirm-project"
terminal(cmd, cwd=AGENT_DIR)
r = subprocess.run(cmd, shell=True, cwd=AGENT_DIR, text=True, capture_output=True)
print(r.stdout[-3500:])
if r.returncode != 0:
    print(r.stderr[-3000:]); raise RuntimeError("deploy failed")

In [ ]:
# --- Record the deployed instance's ids for the next labs ---
def save_to_workshop_env(**kv):
    """Persist values for the next labs (workshop.env) and for this kernel."""
    lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
    for k, v in kv.items():
        lines = [l for l in lines if not l.startswith(f"{k}=")]
        lines.append(f"{k}={v}")
        os.environ[k] = str(v)
    ENV_FILE.write_text("\n".join(lines) + "\n")
    print("saved:", ", ".join(f"{k}={v}" for k, v in kv.items()))

# Read the resource name that agents-cli wrote after the deploy, and derive the id and REST URL from it.
meta = json.loads((AGENT_DIR / "deployment_metadata.json").read_text())
NOVA_AGENT_ENGINE    = meta["remote_agent_runtime_id"]          # projects/NUM/locations/REGION/reasoningEngines/ID
NOVA_AGENT_ENGINE_ID = NOVA_AGENT_ENGINE.split("/")[-1]
NOVA_AGENT_URL       = f"https://{REGION}-aiplatform.googleapis.com/v1/{NOVA_AGENT_ENGINE}"

# Save them to workshop.env. Prefixed NOVA_ on purpose: agents-cli itself reads AGENT_ENGINE_ID / AGENT_RUNTIME_ID from the environment.
save_to_workshop_env(NOVA_AGENT_ENGINE=NOVA_AGENT_ENGINE, NOVA_AGENT_ENGINE_ID=NOVA_AGENT_ENGINE_ID, NOVA_AGENT_URL=NOVA_AGENT_URL)

# Where to see it in the console.
print()
print("Console (deployments):", f"https://console.cloud.google.com/agent-platform/runtimes?project={PROJECT_ID}")
print("Console (this agent): ", f"https://console.cloud.google.com/vertex-ai/agents/agent-engines/locations/{REGION}/agent-engines/{NOVA_AGENT_ENGINE_ID}?project={PROJECT_ID}")

### Look at what was created

The resource description shows the container spec, the environment variables the CLI set
(telemetry on, model endpoint `eu`, prompt content **not** captured in spans by default) and
the service account the agent runs as (`service-…@gcp-sa-aiplatform-re.iam.gserviceaccount.com`).
Open the **Console → this agent** link above: you get a **Playground** tab to chat, plus
Metrics, Sessions and Traces tabs that we'll use in Lab04.

In [ ]:
# --- Inspect the deployed instance through the REST API ---
import requests, google.auth
from google.auth.transport.requests import Request

def gcp_token():
    """Return a short-lived access token of the notebook user for raw REST calls."""
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())
    return creds.token

# Fetch the instance and print the parts that matter: state, identity, env vars, resources and scaling.
engine = requests.get(NOVA_AGENT_URL, headers={"Authorization": f"Bearer {gcp_token()}"}).json()
spec = engine.get("spec", {})
print("display name :", engine["displayName"])
print("state        :", engine.get("state"))
print("identity     :", spec.get("effectiveIdentity"), "  <- the agent's own principal (Agent Identity)")
print("env vars     :", {e["name"]: e.get("value") for e in spec.get("deploymentSpec", {}).get("env", [])})
print("resources    :", spec.get("deploymentSpec", {}).get("resourceLimits"), "| scaling:", {k: v for k, v in spec.get("deploymentSpec", {}).items() if "nstance" in k})

## 2.3 Talk to the deployed agent

### a) From the CLI — `agents-cli run --url … --mode adk`

Same command as in Lab01, pointed at the cloud. Note the numeric **session id** the
runtime returns — that is an Agent Platform Session, stored server-side.

In [ ]:
# --- a) Talk to the deployed agent from the CLI ---
def remote_run(prompt, mode="adk", extra=""):
    """Echo and run `agents-cli run --url <deployed agent>` with one prompt; print the reply."""
    cmd = f'agents-cli run --url {NOVA_AGENT_URL} --mode {mode} {extra} "{prompt}"'.replace("  ", " ")
    terminal(cmd, cwd=AGENT_DIR)
    r = subprocess.run(cmd, shell=True, cwd=AGENT_DIR, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())

# Ask one shopper question; --url points the CLI at the cloud instance instead of a local server.
remote_run("Which gaming monitor do you sell and what does it cost?")

### b) From Python — the Agent Platform SDK

This is what a web shop backend would do: create a session per shopper, stream events,
and reuse the session id on the next request. The **second question** only makes sense with
the history of the first — the runtime keeps it, not your code.

The stream carries **every event** of the turn, not just the answer: the model's tool calls, the
tools' responses and finally the text. The helper below prints them behind a `verbosity` switch.
A shop backend would run it at `0` (answer only) and rely on traces and logs (Lab04) for the
internals — printing tool payloads to end users is for workshops, not production.

In [ ]:
# --- b) Talk to the deployed agent from Python: one session, three turns ---
import asyncio, vertexai

# Get a handle on the deployed instance.
client = vertexai.Client(project=PROJECT_ID, location=REGION)
remote_agent = client.agent_engines.get(name=NOVA_AGENT_ENGINE)

# verbosity: 0 = answer only (what a shop backend would use), 1 = + tool calls, 2 = + tool responses
VERBOSITY = 2

async def ask(user_id, session_id, text, verbosity=None):
    """Stream one turn to the deployed agent; print tool calls / responses per VERBOSITY, then the final answer."""
    verbosity = VERBOSITY if verbosity is None else verbosity
    print(f"\n{'=' * 78}\nUSER  ({user_id}, session {session_id}):\n  {text}\n{'-' * 78}")
    final, steps = None, 0
    async for event in remote_agent.async_stream_query(user_id=user_id, session_id=session_id, message=text):
        for part in event.get("content", {}).get("parts", []):
            if "functionCall" in part:
                steps += 1
                if verbosity >= 1:
                    fc = part["functionCall"]; print(f"  -> tool call     {fc['name']}({json.dumps(fc.get('args', {}))})")
            elif "functionResponse" in part:
                if verbosity >= 2:
                    fr = part["functionResponse"]; print(f"  <- tool response {fr['name']}: {json.dumps(fr.get('response', {}))[:220]}")
            elif "text" in part and not part.get("thought"):
                final = part["text"]
    print(f"{'-' * 78}\nNOVA  ({steps} tool call{'s' if steps != 1 else ''}):\n  {(final or '').strip()}")

# Create a session for one shopper. The runtime stores it in Sessions; the id is generated.
session = await remote_agent.async_create_session(user_id="shopper-42")
SESSION_ID = session["id"]
print("session id:", SESSION_ID)

# Three turns in the same session: the agent remembers the conversation because history lives server-side.
await ask("shopper-42", SESSION_ID, "I am looking for a robot vacuum.")
await ask("shopper-42", SESSION_ID, "How much stock is left of it, and what is the return policy?")
await ask("shopper-42", SESSION_ID, "Thanks! Which one did you recommend again?", verbosity=0)   # answer only, as a backend would print it

In [ ]:
# --- Sessions are first-class resources: list them and inspect one ---
# List the shopper's sessions on the deployed instance.
sessions = await remote_agent.async_list_sessions(user_id="shopper-42")
for s in sessions["sessions"]:
    print("session", s["id"], "| user", s["userId"], "| last update", time.strftime("%H:%M:%S", time.gmtime(s["lastUpdateTime"])))

# Fetch the session we just used and count the events (user messages, tool calls, replies) stored server-side.
detail = await remote_agent.async_get_session(user_id="shopper-42", session_id=SESSION_ID)
print(f"events stored server-side: {len(detail['events'])}")

### c) Over the A2A protocol

Every project scaffolded by the Agents CLI also serves the **Agent2Agent (A2A)** protocol.
Agent Runtime exposes the container's routes through an `/api` passthrough, so the agent
publishes a standard **agent card** and a JSON-RPC endpoint that any A2A client can use: another
agent, Gemini Enterprise, a partner. Let's fetch the card and send one message.

In [ ]:
# --- c) Talk to the deployed agent over the A2A protocol: agent card -> message/send ---
# The runtime exposes the ADK app's A2A endpoint under /api/a2a/<app name>; the same bearer token works.
A2A_BASE = f"https://{REGION}-aiplatform.googleapis.com/reasoningEngines/v1/{NOVA_AGENT_ENGINE}/api/a2a/app"
headers = {"Authorization": f"Bearer {gcp_token()}", "Content-Type": "application/json"}

# Read the agent card: how other agents discover what Nova can do.
card = requests.get(f"{A2A_BASE}/.well-known/agent-card.json", headers=headers).json()
print("agent card:", json.dumps({k: card[k] for k in ("name", "description", "version", "capabilities")}, indent=2))
print("skills    :", [s["name"] for s in card.get("skills", [])])

# Send one message with the JSON-RPC method message/send and read the answer from the returned task's artifacts.
rpc = {"jsonrpc": "2.0", "id": "1", "method": "message/send",
       "params": {"message": {"role": "user", "messageId": "m1",
                              "parts": [{"kind": "text", "text": "What is the return policy for phones? One sentence."}]}}}
task = requests.post(A2A_BASE, headers=headers, json=rpc).json()["result"]
print("\nA2A task state:", task["status"]["state"])
for artifact in task.get("artifacts", []):
    for part in artifact.get("parts", []):
        if part.get("kind") == "text":
            print("Nova (via A2A):", part["text"])

## 2.4 Agent Registry — your agent is already catalogued

**Agent Registry** is the project-level catalogue of agents, MCP servers, skills and
endpoints. Agents deployed to Agent Runtime are registered **automatically** (so are Google's
own remote MCP servers such as BigQuery, in the `global` location). Later, Agent Gateway
policies (Lab06) refer to registry entries, and the console's Topology/Observability views
hang off them.

In [ ]:
# --- Agent Registry: the deployed agent and the Google-managed MCP servers are already listed ---
# List the agents registered in our region (the deploy registered Nova automatically).
cmd = f"gcloud agent-registry agents list --project={PROJECT_ID} --location={REGION} --format='table(name.basename(),displayName,createTime)'"
terminal(cmd); print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

# List a few of the Google-managed MCP servers (registered under location global; we use BigQuery's in Lab03).
cmd = f"gcloud agent-registry mcp-servers list --project={PROJECT_ID} --location=global --format='value(displayName)' | head -8"
terminal(cmd); print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

# Where to see it in the console.
print("Console:", f"https://console.cloud.google.com/agent-platform/agent-registry?project={PROJECT_ID}")

## Recap

* `agents-cli deploy` turned the Lab01 project into a managed **Agent Runtime** instance in `europe-west1`, with the model still served from `eu`.
* You called it via CLI, SDK and **A2A**; the runtime's **Sessions** kept a multi-turn conversation with zero code.
* The agent (and Google's MCP servers) appear in **Agent Registry** automatically.
* Publishing to the employee-facing **Gemini Enterprise** app is one command once a licence is available; the optional [Lab08](lab08_gemini_enterprise.ipynb) shows it.

`workshop.env` now contains `NOVA_AGENT_ENGINE`, `NOVA_AGENT_ENGINE_ID` and `NOVA_AGENT_URL`.

**Next:** [Lab03 — MCP servers, Skills and Agent Registry](lab03_mcp_and_skills.ipynb): real data through Google's BigQuery MCP server, a custom MCP server in the registry, and skills for the analyst.